# Stationarity in Synthetic Examples

In this notebook, we wish to use the Augmented Dickey-Fuller test (ADF) on three synthetic series to test stationarity.

## Setup

Firstly, we construct a purely whitenoise series, a random walk series, and a mean-reverting first order autoregressive model (AR(1)). We expect the whitenoise to be stationary by construction, the random walk to be non-stationary, and the mean-reverting AR(1) to be stationary so long as $|\phi| < 1$.

In [5]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent # parent dir
sys.path.insert(0,str(project_root/"src"))

from statsmodels.tsa.stattools import adfuller
from cointegration import half_life, hurst_exponent
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(314)
n = 5000

white_noise = np.random.normal(0,1,n)

random_walk = np.cumsum(np.random.normal(0,1,n))

phi = 0.7
ar_1 = np.zeros(n, dtype=float)
ar_accum = np.zeros(n, dtype=float)
e_t = np.random.normal(0,1,n)


for i in range(1,n):
    ar_1[i] = phi*ar_1[i-1] + e_t[i]
    ar_accum[i] = ar_1[i] + ar_accum[i-1]


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# ADF Test

Next we run the ADF test on all three time series to verify our predictions.

In [6]:
res_1 = adfuller(white_noise,regression='c')
res_2 = adfuller(random_walk,regression='c')
res_3 = adfuller(ar_1,regression='c')


print("White noise results:")
print(f'p-value: {res_1[1]}')
print(f'ADF Statistic: {res_1[0]}')

print("Random walk results:")
print(f'p-value: {res_2[1]}')
print(f'ADF Statistic: {res_2[0]}')

print("Mean-reverting AR(1) results:")
print(f'p-value: {res_3[1]}')
print(f'ADF Statistic: {res_3[0]}')



White noise results:
p-value: 0.0
ADF Statistic: -71.5362803938862
Random walk results:
p-value: 0.2622579328906582
ADF Statistic: -2.056748456936835
Mean-reverting AR(1) results:
p-value: 0.0
ADF Statistic: -30.16238180604724


Thus we see that both the white noise and mean-reverting AR(1) have p-values less than 0.05 which allows us to reject the null hypothesis and conclude that these series are likely stationary. Furthermore, the random walk has a p-value of 0.26, thus we fail to reject the null hypothesis.

## Half life verification

In this section, we wish to verify the half-life function through synthetic data. Below we generate a series with a specific $\phi$ and verify that the theoretical and empirical bounds match.

In [7]:
phi = 0.9

ar = np.zeros(n)
shocks = np.random.normal(0, 1, n)

for i in range(1,n):
    ar[i] = phi*ar[i-1] + shocks[i]
ar_series = pd.Series(ar)

# Theoretical half-life
theoretical = -np.log(2) / np.log(phi)
print(f"Theoretical half-life for phi={phi}: {theoretical:.2f}")

# Empirical half-life
empirical = half_life(ar_series)
print(f"Empirical half-life: {empirical:.2f}")

Theoretical half-life for phi=0.9: 6.58
Empirical half-life: 6.91


## Hurst Exponent Verification

Below we test the Hurst Exponent function on a random walk, AR(1), and accumulated AR(1) sum. We expect the random walk to be ~ 0.5, the AR(1) to be < 0.5 since it is mean-reverting, and the accumulated AR(1) > 0.5 as it must trend upwards.

In [8]:
h_1 = hurst_exponent(pd.Series(random_walk))
h_2 = hurst_exponent(pd.Series(ar_1))
h_3 = hurst_exponent(pd.Series(ar_accum))

print(f'Random Walk Hurst exponent: {h_1}')
print(f'AR(1) Hurst exponent: {h_2}')
print(f'Accumulated AR(1) Hurst exponent: {h_3}')


Random Walk Hurst exponent: 0.4839908251618521
AR(1) Hurst exponent: 0.051331543942177364
Accumulated AR(1) Hurst exponent: 0.6043917455421829
